In [1]:
pip install onnxruntime tf2onnx tensorflow scikit-learn pandas numpy pillow matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 65.5 MB/s eta 0:00:00


In [2]:
!unzip -q SelectedImages.zip -d .

In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import os, warnings, json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping

import tf2onnx
import onnx

warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────
IMG_SIZE     = 128
PAD_COLOR    = (128, 128, 128)
BATCH_SIZE   = 16
EPOCHS       = 60
PATIENCE     = 10
LR_INIT      = 3e-4
LR_MIN       = 1e-5
WEIGHT_DECAY = 1e-4
DROPOUT_RATE = 0.3
VAL_SPLIT    = 0.20
TEST_SPLIT   = 0.15
RANDOM_SEED  = 42
CLASSES      = ["Blue", "Yellow", "Purple"]

# ── COST MATRIX: penalise Blue↔Purple confusion ───────────────────────────────
COST_MATRIX = np.array([
    [1.0,  1.0,  5.0],
    [1.0,  1.0,  1.0],
    [5.0,  1.0,  1.0],
], dtype=np.float32)


def cost_sensitive_loss(cost_matrix):
    cm_tensor = tf.constant(cost_matrix)

    def loss_fn(y_true, y_pred):
        y_true_int = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred_int = tf.argmax(y_pred, axis=1, output_type=tf.int32)

        indices = tf.stack([y_true_int, y_pred_int], axis=1)
        costs   = tf.gather_nd(cm_tensor, indices)

        ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        return tf.reduce_mean(ce * costs)

    return loss_fn


# ── DATA LOADING ──────────────────────────────────────────────────────────────
def load_dataset(csv_path, images_root):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df = df[["ID", "Label"]].copy()

    df["ID"]    = df["ID"].astype(str).str.strip().astype("object")
    df["Label"] = df["Label"].astype(str).str.strip().astype("object")

    id_to_path = {}
    for root, _, files in os.walk(images_root):
        for fname in files:
            if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                id_to_path[Path(fname).stem] = os.path.join(root, fname)

    df["filepath"] = df["ID"].apply(lambda x: id_to_path.get(x))
    df = df.dropna(subset=["filepath"]).reset_index(drop=True)

    print(f"Loaded {len(df)} images\n{df['Label'].value_counts().to_string()}\n")
    return df


# ── PREPROCESSING ─────────────────────────────────────────────────────────────
def pad_and_resize(img):
    img = img.convert("RGB")
    w, h = img.size
    s = max(w, h)
    padded = Image.new("RGB", (s, s), PAD_COLOR)
    padded.paste(img, ((s - w) // 2, (s - h) // 2))
    return np.array(padded.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS))

def load_and_preprocess(filepath):
    rgb = pad_and_resize(Image.open(filepath)).astype(np.float32) / 255.0
    return tf.image.rgb_to_hsv(rgb[np.newaxis])[0].numpy()

def compute_mean_std(filepaths):
    stack = np.stack([load_and_preprocess(fp) for fp in filepaths])
    mean  = stack.mean(axis=(0, 1, 2))
    std   = stack.std(axis=(0, 1, 2)) + 1e-7
    return mean, std


# ── AUGMENTATION LAYERS ───────────────────────────────────────────────────────
@tf.keras.utils.register_keras_serializable()
class SaturationJitterLayer(layers.Layer):
    def __init__(self, lower=0.85, upper=1.15, **kwargs):
        super().__init__(**kwargs)
        self.lower, self.upper = lower, upper

    def call(self, x, training=None):
        if not training:
            return x
        f = tf.random.uniform((tf.shape(x)[0], 1, 1, 1), self.lower, self.upper)
        h, s, v = x[..., 0:1], x[..., 1:2], x[..., 2:3]
        return tf.concat([h, tf.clip_by_value(s * f, 0.0, 1.0), v], axis=-1)

    def get_config(self):
        return {**super().get_config(), "lower": self.lower, "upper": self.upper}

@tf.keras.utils.register_keras_serializable()
class RandomErasingLayer(layers.Layer):
    def __init__(self, min_frac=0.10, max_frac=0.20, **kwargs):
        super().__init__(**kwargs)
        self.min_frac, self.max_frac = min_frac, max_frac

    def call(self, x, training=None):
        if not training:
            return x
        H = W = IMG_SIZE
        area = tf.cast(H * W, tf.float32)
        ea   = tf.random.uniform((), self.min_frac, self.max_frac) * area
        asp  = tf.random.uniform((), 0.5, 2.0)
        eh   = tf.minimum(tf.cast(tf.sqrt(ea / asp), tf.int32), H - 1)
        ew   = tf.minimum(tf.cast(tf.sqrt(ea * asp), tf.int32), W - 1)
        y0   = tf.random.uniform((), 0, H - eh, dtype=tf.int32)
        x0   = tf.random.uniform((), 0, W - ew, dtype=tf.int32)
        pad  = [[y0, H - y0 - eh], [x0, W - x0 - ew], [0, 0]]
        mask = 1.0 - tf.pad(tf.ones([eh, ew, 3], dtype=x.dtype), pad)
        return x * mask

    def get_config(self):
        return {**super().get_config(), "min_frac": self.min_frac, "max_frac": self.max_frac}


# ── MODEL BUILDERS ────────────────────────────────────────────────────────────
def build_augmentation_pipeline():

    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="aug_input")

    x = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(25/360),
        layers.RandomZoom((-0.20, 0.0)),
    ], name="standard_keras_augs")(inp)

    x = SaturationJitterLayer(name="sat_jitter")(x)
    out = RandomErasingLayer(name="random_erase")(x)

    return keras.Model(inputs=inp, outputs=out, name="AUGMENTATION_PIPELINE")


def build_core_model(num_classes=3, mean=None, std=None):

    l2  = regularizers.L2(WEIGHT_DECAY)
    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="core_input")

    x = inp
    if mean is not None:
        x = layers.Normalization(mean=mean, variance=std**2, name="input_norm")(x)

    for filters, name in [(64, "b1"), (128, "b2"), (256, "b3")]:
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, kernel_regularizer=l2, name=f"conv_{name}")(x)
        x = layers.BatchNormalization(name=f"bn_{name}")(x)
        x = layers.ReLU(name=f"relu_{name}")(x)
        x = layers.AveragePooling2D(2, name=f"pool_{name}")(x)

    x   = layers.GlobalAveragePooling2D(name="gap")(x)
    x   = layers.Dropout(DROPOUT_RATE, name="dropout")(x)
    x   = layers.Dense(64, activation="relu", kernel_regularizer=l2, name="dense_64")(x)
    out = layers.Dense(num_classes, activation="softmax", kernel_regularizer=l2, name="classifier")(x)

    return keras.Model(inputs=inp, outputs=out, name="KHILONA_CORE")


# ── CUSTOM CALLBACK ───────────────────────────────────────────────────────────
class CoreModelCheckpoint(keras.callbacks.Callback):

    def __init__(self, core_model, filepath, monitor="val_loss"):
        super().__init__()
        self.core_model = core_model
        self.filepath = filepath
        self.monitor = monitor
        self.best_loss = np.inf

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current_loss = logs.get(self.monitor)

        if current_loss is not None and current_loss < self.best_loss:
            self.best_loss = current_loss
            self.core_model.save(self.filepath)


# ── DATA PIPELINE ─────────────────────────────────────────────────────────────
def make_dataset(filepaths, labels, training):
    def _load(fp_bytes, label):
        img = load_and_preprocess(fp_bytes.numpy().decode())
        return img, label

    def _tf_load(fp, label):
        img, lbl = tf.py_function(_load, [fp, label], [tf.float32, tf.int32])
        img.set_shape((IMG_SIZE, IMG_SIZE, 3))
        lbl.set_shape(())
        return img, lbl

    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if training:
        ds = ds.shuffle(len(filepaths), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    return ds.map(_tf_load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# ── MAIN ──────────────────────────────────────────────────────────────────────
def main(csv_path, images_root, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    df = load_dataset(csv_path, images_root)
    le = LabelEncoder()
    le.fit(CLASSES)
    y_all = le.transform(df["Label"].values)
    X_all = df["filepath"].astype(object).values
    class_names = le.classes_.tolist()

    X_tv, X_test, y_tv, y_test = train_test_split(
        X_all, y_all, test_size=TEST_SPLIT, stratify=y_all, random_state=RANDOM_SEED)
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=VAL_SPLIT / (1 - TEST_SPLIT), stratify=y_tv, random_state=RANDOM_SEED)

    print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}\n")

    counts = np.bincount(y_train, minlength=len(class_names)).astype(float)
    class_weights = {i: len(y_train) / (len(class_names) * c) for i, c in enumerate(counts)}
    print(f"Class weights: { {class_names[i]: round(w, 3) for i, w in class_weights.items()} }\n")

    mean, std = compute_mean_std(X_train.tolist())
    np.save(os.path.join(output_dir, "norm_mean.npy"), mean)
    np.save(os.path.join(output_dir, "norm_std.npy"),  std)
    np.save(os.path.join(output_dir, "class_names.npy"), np.array(class_names))

    inference_cfg = {
        "img_size":   IMG_SIZE,
        "pad_color":  list(PAD_COLOR),
        "classes":    class_names,
    }
    with open(os.path.join(output_dir, "config.json"), "w") as f:
        json.dump(inference_cfg, f, indent=2)

    train_ds = make_dataset(X_train.tolist(), y_train.astype(np.int32), training=True)
    val_ds   = make_dataset(X_val.tolist(),   y_val.astype(np.int32),   training=False)
    test_ds  = make_dataset(X_test.tolist(),  y_test.astype(np.int32),  training=False)

    aug_model = build_augmentation_pipeline()
    core_model = build_core_model(num_classes=3, mean=mean, std=std)

    train_inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="wrapper_input")
    x = aug_model(train_inp)
    train_out = core_model(x)
    training_wrapper = keras.Model(inputs=train_inp, outputs=train_out, name="TRAINING_WRAPPER")

    print("\n--- Training Wrapper Architecture ---")
    training_wrapper.summary()

    print("\n--- Core Inference Architecture ---")
    core_model.summary()

    total_steps = (len(X_train) // BATCH_SIZE) * EPOCHS
    lr = keras.optimizers.schedules.CosineDecay(LR_INIT, total_steps, alpha=LR_MIN / LR_INIT)
    training_wrapper.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=cost_sensitive_loss(COST_MATRIX),
        metrics=["accuracy"]
    )

    best_path = os.path.join(output_dir, "khilona_best_core.keras")
    history = training_wrapper.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
            CoreModelCheckpoint(core_model=core_model, filepath=best_path, monitor="val_loss")
        ],
        class_weight=class_weights,
        verbose=2
    )

    # ── Training curves ───────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, key, title in zip(axes, ["loss", "accuracy"], ["Loss", "Accuracy"]):
        ax.plot(history.history[key],          label="Train")
        ax.plot(history.history[f"val_{key}"], label="Val")
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "training_history.png"), dpi=150)
    plt.close()

    # ── Evaluation  ───────────────────────────────────────────────────────────
    y_pred = np.argmax(core_model.predict(test_ds, verbose=0), axis=1)
    acc    = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=class_names, digits=4)
    cm     = confusion_matrix(y_test, y_pred)
    cm_df  = pd.DataFrame(cm, index=class_names, columns=class_names)

    print("\n" + "=" * 60)
    print(f"  Test Accuracy: {acc:.4f}  ({acc*100:.2f}%)")
    print("=" * 60)
    print("\nClassification Report:\n")
    print(report)
    print("Confusion Matrix (rows=True, cols=Predicted):")
    print(cm_df.to_string())
    print()

    with open(os.path.join(output_dir, "test_results.txt"), "w") as f:
        f.write(f"Test Accuracy: {acc:.4f}  ({acc*100:.2f}%)\n\n")
        f.write("Classification Report:\n\n" + report + "\n\n")
        f.write("Confusion Matrix (rows=True, cols=Predicted):\n\n" + cm_df.to_string() + "\n\n")

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix — Test Set")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "confusion_matrix.png"), dpi=150)
    plt.close()

    core_model.save(os.path.join(output_dir, "khilona_final_core.keras"))
    print(f"Outputs saved to: {output_dir}\n")


    print("\n--- Converting to ONNX ---")
    onnx_model_path = os.path.join(output_dir, "khilona_model.onnx")

    spec = (tf.TensorSpec((None, IMG_SIZE, IMG_SIZE, 3), tf.float32, name="input"),)

    model_proto, _ = tf2onnx.convert.from_keras(
      core_model,
      input_signature=spec,
      opset=13,
      output_path=onnx_model_path
    )

    print(f"ONNX model saved to: {onnx_model_path}")


# ── ENTRY POINT ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    CSV_FILE_PATH = "selected_images.csv"
    IMAGES_FOLDER = "SelectedImages"
    OUTPUT_FOLDER = "output"

    main(CSV_FILE_PATH, IMAGES_FOLDER, OUTPUT_FOLDER)

Loaded 225 images
Label
Purple    75
Blue      75
Yellow    75

Train: 146  Val: 45  Test: 34

Class weights: {'Blue': np.float64(0.993), 'Purple': np.float64(0.993), 'Yellow': np.float64(1.014)}


--- Training Wrapper Architecture ---


Model: "TRAINING_WRAPPER"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ wrapper_input (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ AUGMENTATION_PIPELINE           │ (None, 128, 128, 3)    │             0 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ KHILONA_CORE (Functional)       │ (None, 3)              │       388,803 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 388,803 (1.48 MB)

 Trainable params: 387,907 (1.48 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ core_input (InputLayer)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ input_norm (Normalization)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_b1 (Conv2D)                │ (None, 128, 128, 64)   │         1,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_b1 (BatchNormalization)      │ (None, 128, 128, 64)   │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_b1 (ReLU)                  │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_b1 (AveragePooling2D)      │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_b2 (Conv2D)                │ (None, 64, 64, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_b2 (BatchNormalization)      │ (None, 64, 64, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_b2 (ReLU)                  │ (None, 64, 64, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_b2 (AveragePooling2D)      │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_b3 (Conv2D)                │ (None, 32, 32, 256)    │       294,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_b3 (BatchNormalization)      │ (None, 32, 32, 256)    │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_b3 (ReLU)                  │ (None, 32, 32, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_b3 (AveragePooling2D)      │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_64 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier (Dense)              │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 388,803 (1.48 MB)

 Trainable params: 387,907 (1.48 MB)

 Non-trainable params: 896 (3.50 KB)

Epoch 1/60
10/10 - 13s - 1s/step - accuracy: 0.5205 - loss: 1.8758 - val_accuracy: 0.7111 - val_loss: 1.5920
Epoch 2/60
10/10 - 2s - 243ms/step - accuracy: 0.6986 - loss: 1.1402 - val_accuracy: 0.6444 - val_loss: 1.6528
Epoch 3/60
10/10 - 4s - 356ms/step - accuracy: 0.6781 - loss: 1.1057 - val_accuracy: 0.8222 - val_loss: 1.3592
Epoch 4/60
10/10 - 3s - 262ms/step - accuracy: 0.6986 - loss: 0.9986 - val_accuracy: 0.9111 - val_loss: 1.0400
Epoch 5/60
10/10 - 3s - 256ms/step - accuracy: 0.8151 - loss: 0.8507 - val_accuracy: 0.8889 - val_loss: 1.0689
Epoch 6/60
10/10 - 2s - 242ms/step - accuracy: 0.7877 - loss: 0.7964 - val_accuracy: 0.9333 - val_loss: 0.9323
Epoch 7/60
10/10 - 3s - 276ms/step - accuracy: 0.7397 - loss: 1.0964 - val_accuracy: 0.9111 - val_loss: 1.0459
Epoch 8/60
10/10 - 5s - 475ms/step - accuracy: 0.6849 - loss: 1.1251 - val_accuracy: 0.8889 - val_loss: 1.0927
Epoch 9/60
10/10 - 3s - 276ms/step - accuracy: 0.7740 - loss: 0.8971 - val_accuracy: 0.9111 - val_loss: 0.8590
Epo